# 01 - Exploration de games.csv

Objectif : comprendre ce que contient le dataset Steam (FronkonGames, Kaggle) avant de le nettoyer.

In [1]:
import sys
sys.path.append('..')   # pour pouvoir importer le dossier src/ depuis notebooks/

import pandas as pd
from src.data_loader import load_games

DATA_PATH = '../data/games.csv'

## 1. Le piège : un décalage d'une colonne

On charge d'abord le fichier « naïvement », sur 3 lignes seulement.

In [2]:
naive = pd.read_csv(DATA_PATH, nrows=3)
naive[['Name', 'Release date', 'Price', 'Tags']]

,Name,Release date,Price,Tags
2539430,"Aug 1, 2023",0 - 0,0,NaN
496350,"Jul 29, 2016",0 - 20000,65,"Adventure,Visual Novel,Anime,Cute"
1034400,"May 6, 2019",0 - 20000,0,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec..."


La colonne `Name` contient une date, et `Price` vaut 65 pour un jeu à 5,24 $ : tout est décalé d'une colonne.
On compte les noms de l'en-tête et les valeurs d'une ligne :

In [3]:
with open(DATA_PATH, encoding='utf-8') as f:
    header = f.readline()
print('Noms dans l\'en-tête :', len(header.split(',')))
print('Valeurs dans une ligne :', len(naive.columns) + 1, '(colonnes + index)')
print([name for name in header.split(',') if 'DLC' in name])

Noms dans l'en-tête : 39
Valeurs dans une ligne : 40 (colonnes + index)
['DiscountDLC count']


**Explication** : l'en-tête a 39 noms mais chaque ligne a 40 valeurs, car `Discount` et `DLC count` sont collés en `DiscountDLC count`.
pandas prend alors la valeur en trop (AppID) comme index, et chaque nom se retrouve au-dessus de la colonne d'à côté.
**Correction** : `load_games` (dans `src/data_loader.py`) ignore l'en-tête et donne la liste des 40 bons noms avec le paramètre `names` de `pd.read_csv`.

## 2. Chargement corrigé (cellule lente : à ne lancer qu'une fois)

In [4]:
USEFUL_COLUMNS = ['AppID', 'Name', 'Release date', 'Estimated owners',
                  'Price', 'Positive', 'Negative', 'Tags']
df = load_games(DATA_PATH, usecols=USEFUL_COLUMNS)
print(df.shape)
df.columns

(125855, 8)


Index(['AppID', 'Name', 'Release date', 'Estimated owners', 'Price',
       'Positive', 'Negative', 'Tags'],
      dtype='str')

In [5]:
df[['Name', 'Price', 'Positive', 'Negative', 'Tags']].head()

,Name,Price,Positive,Negative,Tags
0,Black Dragon Mage Playtest,0.00,0,0,NaN
1,Supipara - Chapter 1 Spring Has Come!,5.24,252,3,"Adventure,Visual Novel,Anime,Cute"
2,Mystery Solitaire The Black Raven,4.99,21,3,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec..."
3,버튜버 파라노이아 - Vtuber Paranoia,8.99,0,0,NaN
4,Maze Quest VR,4.99,0,0,NaN


Le check est bon : un nom de jeu, un prix en dollars, deux nombres d'avis et des tags. Environ 125 000 lignes.

## 3. Types et valeurs manquantes

In [6]:
df.dtypes

AppID                 int64
Name                    str
Release date            str
Estimated owners        str
Price               float64
Positive              int64
Negative              int64
Tags                    str
dtype: object

In [7]:
df.isna().sum()

AppID                   0
Name                    1
Release date            0
Estimated owners        0
Price                   0
Positive                0
Negative                0
Tags                42502
dtype: int64

Les nombres (`Price`, `Positive`, `Negative`) sont déjà numériques. La date et les owners sont du texte : il faudra les convertir au nettoyage.
Environ un jeu sur trois n'a aucun tag : ces jeux ne pourront pas servir de comparables.

## 4. Combien de jeux gratuits ?

In [8]:
n_free = (df['Price'] == 0).sum()
print(n_free, 'jeux gratuits, soit', round(100 * n_free / len(df), 1), '%')

26661 jeux gratuits, soit 21.2 %


Environ 21 % des jeux sont gratuits. Sans prix, on ne peut pas estimer de revenu de vente : ils seront retirés.

## 5. Quelle distribution des prix ?

In [9]:
paid = df[df['Price'] > 0]
paid['Price'].describe()

count    99194.000000
mean         6.104304
std         13.784581
min          0.490000
25%          1.490000
50%          3.590000
75%          6.990000
max        999.980000
Name: Price, dtype: float64

In [10]:
price_bins = [0, 5, 10, 15, 20, 30, 1000]
pd.cut(paid['Price'], bins=price_bins).value_counts().sort_index()

Price
(0, 5]        66830
(5, 10]       20070
(10, 15]       6742
(15, 20]       2977
(20, 30]       1431
(30, 1000]     1144
Name: count, dtype: int64

La grande majorité des jeux payants coûte moins de 10 $. Le prix médian d'un jeu payant est bas : le marché est dominé par de petits jeux.

## 6. Combien de jeux ont moins de 10 avis ?

In [11]:
n_reviews = df['Positive'] + df['Negative']
few = (n_reviews < 10).sum()
print(few, 'jeux ont moins de 10 avis, soit', round(100 * few / len(df), 1), '%')

69193 jeux ont moins de 10 avis, soit 55.0 %


**Chiffre frappant** : plus de la moitié des jeux Steam (55 %) ont moins de 10 avis. La plupart des jeux passent presque inaperçus.

## 7. Combien de jeux sortent par an ?

In [12]:
release = pd.to_datetime(df['Release date'], format='%b %d, %Y', errors='coerce')
print('Dates illisibles :', release.isna().sum())
release.dt.year.value_counts().sort_index().tail(12)

Dates illisibles : 0


Release date
2015     2515
2016     4141
2017     5922
2018     7461
2019     7243
2020     8804
2021    11067
2022    12288
2023    14598
2024    20033
2025    24973
2026     3316
Name: count, dtype: int64

Les sorties explosent : environ 7 000 jeux par an vers 2018, et près de 25 000 en 2025. La concurrence augmente très vite. L'année 2026 est incomplète (le dataset s'arrête avant la fin de l'année).

## 8. Quels sont les 20 tags les plus fréquents ?

In [13]:
# Les tags d'un jeu sont dans une seule chaîne : on la découpe en liste (split),
# puis explode crée une ligne par tag, qu'on peut enfin compter.
tags = df['Tags'].str.split(',').explode()
tags.value_counts().head(20)

Tags
Singleplayer      50353
Indie             48556
Action            36812
Casual            36565
Adventure         35200
2D                26794
3D                17997
Simulation        17896
Strategy          17850
Puzzle            16370
Atmospheric       16284
RPG               15745
Colorful          14261
Pixel Graphics    13869
Story Rich        13345
Exploration       13241
Cute              12680
First-Person      12295
Fantasy           11501
Arcade            11495
Name: count, dtype: int64

`Indie` est le 2e tag le plus fréquent (presque 50 000 jeux) : le segment indé est massif et très concurrentiel. Les tags de genre (Action, Casual, Adventure) dominent ; les niches plus précises (Roguelike, Deckbuilding...) seront étudiées à l'étape 7.

## 9. Les owners (estimation SteamSpy)

In [14]:
df['Estimated owners'].value_counts().head(8)

Estimated owners
0 - 20000            75705
0 - 0                24579
20000 - 50000        11397
50000 - 100000        5355
100000 - 200000       3454
200000 - 500000       2856
500000 - 1000000      1155
1000000 - 2000000      729
Name: count, dtype: int64

Les owners sont donnés en fourchettes de texte (« 20000 - 50000 »). 80 % des jeux sont dans « 0 - 20000 » ou « 0 - 0 » : la fourchette est trop large pour estimer des ventes, d'où la méthode par les avis (étape 5).